In [ ]:
!which python

In [ ]:
import fiftyone
import fiftyone.utils.random
import pathlib
import yaml
from ultralytics import YOLO


# set up the paths, names and settings

In [ ]:
# select to models parent folder
models_parent_dir = pathlib.Path.cwd().parent / "models"
datasets_parent_dir = pathlib.Path.cwd().parent / "datasets"

# choose a name for the new merged dataset
merged_dataset_name = "der_merger-200"

# Load settings
with open('settings.yaml') as f:
    settings = yaml.safe_load(f)

splits = ["train", "test", "val"]

# setup the datasets in fiftyone

In [ ]:
datasets_to_reset = [merged_dataset_name, "tmp_dataset"]

for dataset in datasets_to_reset:
    try:
        fiftyone.delete_dataset(dataset)
        print(f"Deleted existing dataset: {dataset}")
    except:
        print(f"No existing dataset named: {dataset}")

tmp_dataset = fiftyone.Dataset("tmp_dataset")

# choose a new dataset name to combine the others in
merged_dataset = fiftyone.Dataset(merged_dataset_name)

# choose a dataset to add the predictions to
source_dataset = fiftyone.load_dataset("coco-2017-train-validation-test-200")
tmp_dataset.add_samples(source_dataset)

# datasets_to_add = [fiftyone.load_dataset("ebi-fused-4")]
datasets_to_add = [fiftyone.load_dataset("ebi-fused-4").take(200)]

Deleted existing dataset: der_merger
Deleted existing dataset: tmp_dataset
 100% |█████████████| 54024/54024 [2.6m elapsed, 0s remaining, 1.9K samples/s]       


# model inferencing on a particular subset

In [29]:
# choose models to inference with, they must match the settings.yaml
chosen_models = ["doors", "windows", "lights"]

conditions = []

# Apply all models to same field, then filter once
for model_name in chosen_models:
    model = YOLO(models_parent_dir / model_name / "best.pt")
    tmp_dataset.apply_model(model, label_field=model_name)

    for class_name, threshold in settings["models"][model_name].items():
        conditions.append(
            (fiftyone.ViewField("label") == class_name) & 
            (fiftyone.ViewField("confidence") >= threshold)
        )
    if conditions:
        keep_condition = conditions[0]
        for condition in conditions[1:]:
            keep_condition |= condition
        filtered_view = tmp_dataset.filter_labels(model_name, keep_condition)
        filtered_view.merge_labels(in_field=model_name, out_field="ground_truth")


# Delete predictions field from the original dataset
# tmp_dataset.delete_sample_fields("predictions")
tmp_dataset.save()

 100% |█████████████| 54024/54024 [10.0m elapsed, 0s remaining, 86.7 samples/s]      
 100% |█████████████| 54024/54024 [10.1m elapsed, 0s remaining, 85.4 samples/s]      
 100% |█████████████| 54024/54024 [10.2m elapsed, 0s remaining, 85.1 samples/s]      


In [30]:
# session = fiftyone.launch_app(tmp_dataset)

# merge datasets

In [31]:
datasets_to_add.append(tmp_dataset)


for dataset in datasets_to_add:
    print(f"adding {len(dataset)} samples from {dataset.name}")
    merged_dataset.merge_samples(dataset)

merged_dataset.save()
print(merged_dataset)

adding 3398 samples from ebi-fused-4
adding 54024 samples from tmp_dataset
Name:        der_merger
Media type:  image
Num samples: 57422
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    doors:            fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    windows:          fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    lights:           fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)


# some tag renaming

In [32]:
# Find all samples tagged with "validation" and retag them as "val"
validation_samples = merged_dataset.match_tags("validation")
validation_samples.tag_samples("val")
validation_samples.untag_samples("validation")

# redistribute splits

In [33]:

print(f"Splits before redistribution: {merged_dataset.count_sample_tags()}")
merged_dataset.untag_samples(splits)
fiftyone.utils.random.random_split(merged_dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(f"Splits after redistribution: {merged_dataset.count_sample_tags()}")

Splits before redistribution: {'test': 41013, 'train': 15494, 'val': 915}
Splits after redistribution: {'train': 40195, 'val': 5742, 'test': 11485}


# some class renaming

In [ ]:
# Rename class labels in ground_truth.detections.label
merged_dataset.update_values(
    "ground_truth.detections.label",
    {"chair": "Chair", "window": "Window", "light": "Light"}
)

In [34]:
merged_dataset.set_values(
    "ground_truth.detections.label",
    "Chair",
    fiftyone.ViewField("ground_truth.detections.label") == "chair"
)
merged_dataset.set_values(
    "ground_truth.detections.label",
    "Window",
    fiftyone.ViewField("ground_truth.detections.label") == "window"
)
merged_dataset.set_values(
    "ground_truth.detections.label",
    "Light",
    fiftyone.ViewField("ground_truth.detections.label") == "light"
)

In [35]:


export_dir = datasets_parent_dir / "multiclass" / f"{merged_dataset_name}-{len(merged_dataset)}"

# All splits must use the same classes list
classes = ["Chair", "Window", "Light", "Door"]


# Export the splits
for split in splits:
    split_view = merged_dataset.match_tags(split)
    split_view.export(
        export_dir=str(export_dir),
        dataset_type=fiftyone.types.YOLOv5Dataset,
        label_field="ground_truth",
        # overwrite=True,
        split=split,
        classes=classes,
        progress=True,
    )

Directory '/home/rolf/GIT/onemanstreasure/datasets/multiclass/der_merger-57422' already exists; export will be merged with existing files
   7% |-------------|  2629/40195 [3.0s elapsed, 46.5s remaining, 379.1 samples/s]    

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'giraffe' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'snowboard' not in provided classes
  warnings.warn(msg)


   7% |\------------|  2779/40195 [3.5s elapsed, 52.8s remaining, 291.8 samples/s]    

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'toilet' not in provided classes
  warnings.warn(msg)
/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'skis' not in provided classes
  warnings.warn(msg)


   8% |█\-----------|  3143/40195 [4.7s elapsed, 1.0m remaining, 298.9 samples/s]     

...

   9% |█------------|  3801/40195 [7.1s elapsed, 1.2m remaining, 286.3 samples/s]     

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'toaster' not in provided classes
  warnings.warn(msg)


  11% |█|-----------|  4291/40195 [8.9s elapsed, 1.4m remaining, 265.0 samples/s]     

/home/rolf/anaconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'cow' not in provided classes
  warnings.warn(msg)


 100% |█████████████| 40195/40195 [1.1m elapsed, 0s remaining, 1.1K samples/s]        
Directory '/home/rolf/GIT/onemanstreasure/datasets/multiclass/der_merger-57422' already exists; export will be merged with existing files
 100% |█████████████| 11485/11485 [17.3s elapsed, 0s remaining, 1.1K samples/s]       
Directory '/home/rolf/GIT/onemanstreasure/datasets/multiclass/der_merger-57422' already exists; export will be merged with existing files
 100% |███████████████| 5742/5742 [9.3s elapsed, 0s remaining, 1.2K samples/s]        
